Baseline + SMOTE + StandartScaler
Model: RandomForest
Podzial: 60/40
Random state = 30

Precision (fraud): 0.9415
Recall (fraud):    0.6550
F1 (fraud):        0.7725

In [61]:
import pandas as pd
import numpy as np
from pandas.core.common import random_state
from sklearn.metrics import roc_auc_score, confusion_matrix, average_precision_score, classification_report

from xgboost import XGBClassifier

In [62]:
df = pd.read_csv("../data/credit_card_frauds.csv")
print(df.head())
print(df.shape)
print(df.info())
print(df.isnull().sum())

  trans_date_trans_time                   merchant       category     amt  \
0   2019-01-01 00:00:44  Heller, Gutmann and Zieme    grocery_pos  107.23   
1   2019-01-01 00:00:51             Lind-Buckridge  entertainment  220.11   
2   2019-01-01 00:07:27                  Kiehn Inc    grocery_pos   96.29   
3   2019-01-01 00:09:03                Beier-Hyatt   shopping_pos    7.77   
4   2019-01-01 00:21:32                 Bruen-Yost       misc_pos    6.85   

                       city state      lat      long  city_pop  \
0                    Orient    WA  48.8878 -118.2105       149   
1                Malad City    ID  42.1808 -112.2620      4154   
2                   Grenada    CA  41.6125 -122.5258       589   
3  High Rolls Mountain Park    NM  32.9396 -105.8189       899   
4                   Freedom    WY  43.0172 -111.0292       471   

                                 job         dob  \
0  Special educational needs teacher  1978-06-21   
1        Nature conservation officer

In [63]:
print(df.describe(include="all").T)
print(df["is_fraud"].value_counts())
print(df["is_fraud"].value_counts(normalize=True))

for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

                          count  unique                               top  \
trans_date_trans_time    339607  338504               2019-12-06 20:30:44   
merchant                 339607     693                       Kilback LLC   
category                 339607      14                     gas_transport   
amt                    339607.0     NaN                               NaN   
city                     339607     176                           Phoenix   
state                    339607      13                                CA   
lat                    339607.0     NaN                               NaN   
long                   339607.0     NaN                               NaN   
city_pop               339607.0     NaN                               NaN   
job                      339607     163                Surveyor, minerals   
dob                      339607     187                        1987-04-23   
trans_num                339607  339607  14392d723bb7737606b2700ac791b7aa   

In [64]:
# 1. Konwersja czasu i sortowanie chronologiczne calego zbioru
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
df = df.sort_values("trans_date_trans_time").reset_index(drop=True)

# 2. Tymczasowy identyfikator klienta
df["client_id"] = (
    df["lat"].astype(str) + "_" +
    df["long"].astype(str) + "_" +
    df["dob"].astype(str) + "_" +
    df["job"].astype(str)
)

# 3. Cechy czasowe wzgledem poprzednich transakcji tego samego klienta
df["time_since_last_trans"] = (
    df.groupby("client_id")["trans_date_trans_time"]
    .diff()
    .dt.total_seconds()
    .fillna(-1)
)

# Czas od jeszcze wczesniejszej transakcji — wykrywa serie szybkich prób
df["time_since_prev2_trans"] = (
    df.groupby("client_id")["trans_date_trans_time"]
    .diff(2)
    .dt.total_seconds()
    .fillna(-1)
)

# 4. Cechy kwotowe wzgledm dotychczasowej historii klienta
# Srednia kroczaca z ostatnich 10 transakcji klienta przed bieżacą transakcja
rolling_mean_10 = (
    df.groupby("client_id")["amt"]
    .transform(lambda s: s.shift(1).rolling(10, min_periods=1).mean())
)
# Jeśli to pierwsza transakcja klienta podstawiamy biezaca kwote
rolling_mean_10 = rolling_mean_10.fillna(df["amt"])

# Kroczace odchylenie standardowe z ostatnich 10 transakcji klienta
rolling_std_10 = (
    df.groupby("client_id")["amt"]
    .transform(lambda s: s.shift(1).rolling(10, min_periods=2).std())
    .fillna(0.0)
)

# Stosunek bieaącej kwoty do seedniej klienta oraz Z-score
df["amt_to_client_mean_10"] = df["amt"] / (rolling_mean_10 + 1e-5)
df["amt_zscore_client_10"] = (df["amt"] - rolling_mean_10) / (rolling_std_10 + 1.0)

# Roznica kwoty względem bezposrednio poprzedniej transakcji klienta
prev_amt = df.groupby("client_id")["amt"].shift(1).fillna(df["amt"])
df["amt_diff_from_prev"] = df["amt"] - prev_amt

# 5. Przemieszczanie się klienta miedzy transakcjami
def haversine_vec(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1_r, lon1_r = np.radians(lat1), np.radians(lon1)
    lat2_r, lon2_r = np.radians(lat2), np.radians(lon2)
    dlat = lat2_r - lat1_r
    dlon = lon2_r - lon1_r
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1_r) * np.cos(lat2_r) * np.sin(dlon / 2.0)**2
    return 2 * R * np.arcsin(np.sqrt(a))

prev_merch_lat = df.groupby("client_id")["merch_lat"].shift(1).fillna(df["merch_lat"])
prev_merch_long = df.groupby("client_id")["merch_long"].shift(1).fillna(df["merch_long"])

# Odleglosc miedzy sprzedawca z poprzedniej transakcji a obecnym sprzedawcą
df["dist_from_prev_merch"] = haversine_vec(
    prev_merch_lat, prev_merch_long,
    df["merch_lat"], df["merch_long"]
)

# Predkosc przemieszczania się miedzy transakcjami
hours_diff = np.where(df["time_since_last_trans"] > 0, df["time_since_last_trans"] / 3600.0, np.nan)
df["travel_speed_kmh"] = (df["dist_from_prev_merch"] / hours_diff).fillna(0.0)
# Przyciecie ekstremalnych wartosci
df["travel_speed_kmh"] = np.clip(df["travel_speed_kmh"], 0, 2000)

# Ustawiamy czas jako indeks tymczasowy do obliczen okien czasowych
df_temp = df.set_index("trans_date_trans_time")

# Liczba transakcji klienta w ciagu ostatniej 1h, 24h i 7 dni
df_temp = df[["client_id", "trans_date_trans_time", "amt"]].copy()
df_temp["orig_idx"] = df.index
df_temp = df_temp.sort_values(["client_id", "trans_date_trans_time"])

df_indexed = df_temp.set_index("trans_date_trans_time")

df_temp["client_tx_count_1h"] = (
    df_indexed.groupby("client_id")["amt"].rolling("1h", closed="left").count().fillna(0).values
)
df_temp["client_tx_count_24h"] = (
    df_indexed.groupby("client_id")["amt"].rolling("24h", closed="left").count().fillna(0).values
)
df_temp["client_tx_count_7d"] = (
    df_indexed.groupby("client_id")["amt"].rolling("7D", closed="left").count().fillna(0).values
)
df_temp["client_amt_sum_24h"] = (
    df_indexed.groupby("client_id")["amt"].rolling("24h", closed="left").sum().fillna(0).values
)

# Przywracamy pierwotną kolejność wierszy przed przypisaniem do df
df_temp = df_temp.sort_values("orig_idx")

df["client_tx_count_1h"] = df_temp["client_tx_count_1h"].values
df["client_tx_count_24h"] = df_temp["client_tx_count_24h"].values
df["client_amt_sum_24h"] = df_temp["client_amt_sum_24h"].values

# 6. Usuwamy tymczasowy identyfikator klienta
df = df.drop(["client_id"], axis=1)

In [65]:
#Pozbywamy sie numeru transakcji
X = df.drop(["is_fraud", "trans_num"], axis=1)
y = df["is_fraud"]

In [66]:
#Stosujemy stratify by zachowac balans klas w zbiorach.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.40,
    random_state=30
)

In [67]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\ny_train:")
print(y_train.value_counts())

print("\ny_test:")
print(y_test.value_counts())

X_train: (203764, 23)
X_test: (135843, 23)

y_train:
is_fraud
0    202695
1      1069
Name: count, dtype: int64

y_test:
is_fraud
0    135130
1       713
Name: count, dtype: int64


In [68]:
#rozbicie daty na godziny, dni, miesiace i dzien tygodnia oraz liczymy wiek.
X_train = X_train.copy()
X_test = X_test.copy()

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = np.sin(dlat / 2.0)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

for data in [X_train, X_test]:
    # 1. Konwersja dat
    data["trans_date_trans_time"] = pd.to_datetime(data["trans_date_trans_time"])
    data["dob"] = pd.to_datetime(data["dob"])

    # 2. Podstawowe cechy czasowe i wiek
    data["trans_hour"] = data["trans_date_trans_time"].dt.hour
    data["trans_day"] = data["trans_date_trans_time"].dt.day
    data["trans_month"] = data["trans_date_trans_time"].dt.month
    data["trans_dayofweek"] = data["trans_date_trans_time"].dt.dayofweek
    data["age"] = data["trans_date_trans_time"].dt.year - data["dob"].dt.year

    # 3. Cykliczne kodowanie czasu (sinus / cosinus)
    data["hour_sin"] = np.sin(2 * np.pi * data["trans_hour"] / 24.0)
    data["hour_cos"] = np.cos(2 * np.pi * data["trans_hour"] / 24.0)

    data["dow_sin"] = np.sin(2 * np.pi * data["trans_dayofweek"] / 7.0)
    data["dow_cos"] = np.cos(2 * np.pi * data["trans_dayofweek"] / 7.0)

    data["month_sin"] = np.sin(2 * np.pi * data["trans_month"] / 12.0)
    data["month_cos"] = np.cos(2 * np.pi * data["trans_month"] / 12.0)

    # 4. Flagi behawioralne czasu (noc oraz weekend)
    data["is_night"] = ((data["trans_hour"] >= 23) | (data["trans_hour"] <= 5)).astype(int)
    data["is_weekend"] = (data["trans_dayofweek"] >= 5).astype(int)

    # 5. Odległość miedzy klientem a sprzedawca
    data["distance_km"] = haversine_distance(
        data["lat"], data["long"],
        data["merch_lat"], data["merch_long"]
    )

    # 6. Logarytm z kwoty transakcji
    data["log_amt"] = np.log1p(data["amt"])

# Usuniecie surowych kolumn dat
X_train = X_train.drop(["trans_date_trans_time", "dob"], axis=1)
X_test = X_test.drop(["trans_date_trans_time", "dob"], axis=1)

print("X_train: ", X_train.shape)
print(X_train.dtypes)

X_train:  (203764, 36)
merchant                   object
category                   object
amt                       float64
city                       object
state                      object
lat                       float64
long                      float64
city_pop                    int64
job                        object
merch_lat                 float64
merch_long                float64
time_since_last_trans     float64
time_since_prev2_trans    float64
amt_to_client_mean_10     float64
amt_zscore_client_10      float64
amt_diff_from_prev        float64
dist_from_prev_merch      float64
travel_speed_kmh          float64
client_tx_count_1h        float64
client_tx_count_24h       float64
client_amt_sum_24h        float64
trans_hour                  int32
trans_day                   int32
trans_month                 int32
trans_dayofweek             int32
age                         int32
hour_sin                  float64
hour_cos                  float64
dow_sin                  

In [69]:
for col in X_train.select_dtypes(include="object").columns:
    print(f"{col}: {X_train[col].nunique()}")

merchant: 693
category: 14
city: 176
state: 13
job: 163


In [70]:
from sklearn.preprocessing import StandardScaler

categorical_columns = [
    "merchant",
    "category",
    "city",
    "state",
    "job"
]

numeric_columns = [col for col in X_train.columns if col not in categorical_columns]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_columns] = scaler.fit_transform(
    X_train[numeric_columns]
)

X_test_scaled[numeric_columns] = scaler.transform(
    X_test[numeric_columns]
)

In [71]:
from imblearn.over_sampling import SMOTENC

categorical_columns = [
    "merchant",
    "category",
    "city",
    "state",
    "job"
]

smotenc = SMOTENC(
    categorical_features=categorical_columns,
    sampling_strategy="auto",
    random_state=30
)

X_train_smote, y_train_smote = smotenc.fit_resample(
    X_train_scaled,
    y_train
)

In [72]:
#Zamiana wartosci tekstowych na liczbowe
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)

X_train_encoded = encoder.fit_transform(X_train_smote[categorical_columns])

X_test_encoded = encoder.transform(X_test[categorical_columns])

In [73]:
from scipy.sparse import hstack

X_train_numeric = X_train_smote[numeric_columns]
X_test_numeric = X_test_scaled[numeric_columns]

X_train_final = hstack([X_train_numeric, X_train_encoded])
X_test_final = hstack([X_test_numeric, X_test_encoded])

In [74]:
from collections import Counter

print("Przed SMOTENC:", Counter(y_train))
print("Po SMOTENC:", Counter(y_train_smote))

print("Train przed:", len(y_train))
print("Train po:", len(y_train_smote))

print("Test:", len(y_test))

Przed SMOTENC: Counter({0: 202695, 1: 1069})
Po SMOTENC: Counter({0: 202695, 1: 202695})
Train przed: 203764
Train po: 405390
Test: 135843


In [75]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    random_state=30,
    n_jobs=-1
)
model.fit(X_train_final, y_train_smote)

RandomForestClassifier(n_jobs=-1, random_state=30)

In [76]:
y_pred = model.predict(X_test_final)
y_pred_proba = model.predict_proba(X_test_final)[:, 1]

In [77]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    135130
           1       0.92      0.73      0.81       713

    accuracy                           1.00    135843
   macro avg       0.96      0.87      0.91    135843
weighted avg       1.00      1.00      1.00    135843



In [78]:
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Precision - jaka część transakcji oznaczonych przez model jako fraud serio nim byla
#
# Recall - jaka część wszystkich rzeczywistych fraudów
#          została wykryta przez model.
#
# F1-score - średnia harmoniczna precision i recall
#
# ROC-AUC - miara zdolności modelu do rozróżniania klas
#
# PR-AUC - pole pod krzywą Precision-Recall
#
# Confusion Matrix:
# TN - poprawnie rozpoznane transakcje normalne
# FP - normalne transakcje błędnie oznaczone jako fraud
# FN - fraudy, których model nie wykrył
# TP - poprawnie wykryte fraudy

report = classification_report(
    y_test,
    y_pred,
    target_names=["Normalna transakcja", "Fraud"],
    digits=4
)

print("                 WYNIKI MODELU RandomForest")

print("\n--- Classification Report ---")
print(report)

# ROC-AUC
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("--- ROC-AUC ---")
print(f"{roc_auc:.4f}")


# PR-AUC
pr_auc = average_precision_score(y_test, y_pred_proba)

print("\n--- PR-AUC ---")
print(f"{pr_auc:.4f}")


# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print("\n--- Confusion Matrix ---")
print(cm)

print("\nInterpretacja:")
print(f"  TN = {tn:>6}  | poprawnie rozpoznane normalne")
print(f"  FP = {fp:>6}  | normalne oznaczone jako fraud")
print(f"  FN = {fn:>6}  | niewykryte fraudy")
print(f"  TP = {tp:>6}  | poprawnie wykryte fraudy")

                 WYNIKI MODELU RandomForest

--- Classification Report ---
                     precision    recall  f1-score   support

Normalna transakcja     0.9986    0.9997    0.9991    135130
              Fraud     0.9174    0.7321    0.8144       713

           accuracy                         0.9982    135843
          macro avg     0.9580    0.8659    0.9067    135843
       weighted avg     0.9982    0.9982    0.9982    135843

--- ROC-AUC ---
0.9952

--- PR-AUC ---
0.8924

--- Confusion Matrix ---
[[135083     47]
 [   191    522]]

Interpretacja:
  TN = 135083  | poprawnie rozpoznane normalne
  FP =     47  | normalne oznaczone jako fraud
  FN =    191  | niewykryte fraudy
  TP =    522  | poprawnie wykryte fraudy


Zbiór danych rozszerzono o te same cechy bezstanowe i behawioralne (m.in. odległość Haversine, czas od poprzednich transakcji, prędkość przemieszczania się, relację kwoty do średniej klienta oraz kroczące agregacje w oknach 1h i 24h), poddane standaryzacji przed generowaniem syntetycznych próbek metodą SMOTENC. Nowe zmienne kontekstowe znacząco zwiększyły skuteczność lasu losowego w warunkach silnego niezrównoważenia klas — F1-score wzrósł z 0,7725 do 0,8144, PR-AUC z 0,8251 do 0,8924, a wykrywalność oszustw (Recall) poprawiła się o ponad 7,7 p.p. (z 65,50% do 73,21%).